# Creazione del debug dataset (episodi bilanciati per esito)

**Scopo.** Generare `datasets/debug_dataset_full_ep.pkl`: un insieme di episodi
completi **bilanciato per esito terminale** (20 arrived / 20 collided /
20 off-road / 20 timeout), campionato dal rollout di un agente salvato.
È il dataset fisso usato dai log di validazione del reward model
(`debug_dataset` in `train_hybrid_sac.py`) e dall'analisi
`reward_model_scatter_analysis.ipynb`.

**Provenienza.** Questo notebook documenta come i file
`debug_dataset*.pkl` già presenti in `datasets/` (scaricati da HF con
`scripts/download_datasets.py`) sono stati generati. Rieseguirlo produce un
NUOVO dataset (rollout diverso): non sovrascrive mai i file esistenti.

**Ambiente.** Richiede `human_feedback_rl`, `stable_baselines3` e
`sumo_rl_ego`/`traci` (apre l'env SUMO): va eseguito sulla macchina di
training, dalla cartella `notebooks/`.

In [ ]:
from pathlib import Path
import pickle

import numpy as np
from stable_baselines3 import SAC, PPO

import sumo_rl_ego as sre
from human_feedback_rl.common.trajectory_generators import TrajectoryGeneratorFromAgent
from human_feedback_rl.common.status import STATUS_ARRIVED, STATUS_COLLIDED, STATUS_OFFROAD, STATUS_TIMEOUT

from loadings import load_reward_ensemble

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = REPO / "datasets"

## 1. Configurazione

Indica il checkpoint da cui campionare il rollout: una run finale
(`outputs/final/<group>/<run>`) o un trial Optuna
(`outputs/optuna/hybrid_sac_<arm>/trial_NNNN/<run>/checkpoint_NNNN`).

In [ ]:
CKPT_DIR = REPO / "outputs" / "final" / "pref_soft" / "pref_soft-seed1" / "checkpoint_0050"
N_ROLLOUT_STEPS = 100_000   # transizioni da campionare
N_PER_CLASS = 20            # episodi per esito terminale
OUT_PATH = DATA_DIR / "debug_dataset_full_ep_new.pkl"   # mai sovrascrivere gli originali

for p in (CKPT_DIR / "reward_model.pt", CKPT_DIR / "agent.zip"):
    assert p.exists(), f"Manca: {p}"
assert not OUT_PATH.exists(), f"{OUT_PATH} esiste gia': scegli un altro nome."

## 2. Env SUMO, agente e reward model

In [ ]:
# Monkeypatch per usare traci nel notebook (una simulazione per processo).
import sumo_gym_ego.core.simulation as _sim_mod
import traci as _traci_mod
_sim_mod.load_traci = lambda use_gui: _traci_mod

env = sre.make_vec_env("HighwayEgo-v0", n_envs=1, base_seed=0,
                       ego="continuous", reward="fast")

reward_model = load_reward_ensemble(CKPT_DIR / "reward_model.pt",
                                    env.observation_space, env.action_space)

def load_agent(path, env):
    for Algo in (SAC, PPO):
        try:
            agent = Algo.load(path, env=env, device="cpu")
            print(f"Agente caricato come {Algo.__name__}")
            return agent
        except Exception:
            continue
    raise RuntimeError(f"Impossibile caricare {path} come SAC o PPO")

agent = load_agent(CKPT_DIR / "agent.zip", env)
generator = TrajectoryGeneratorFromAgent(agent=agent, reward_model=reward_model, venv=env)

## 3. Rollout e distribuzione degli esiti

In [ ]:
trajectories = generator.sample(N_ROLLOUT_STEPS)

terminal = [int(np.argmax(traj[-1].next_status)) for traj in trajectories]
names = {STATUS_ARRIVED: "arrived", STATUS_COLLIDED: "collided",
         STATUS_OFFROAD: "off_road", STATUS_TIMEOUT: "timeout"}
mean_len = np.mean([len(t) for t in trajectories])
print(f"Episodi: {len(trajectories)} | lunghezza media {mean_len:.1f}")
for sid, name in names.items():
    n = sum(s == sid for s in terminal)
    print(f"  {name:8s}: {n:4d} ({100 * n / len(trajectories):.1f}%)")

## 4. Bilanciamento per esito

Primi `N_PER_CLASS` episodi per ciascuno dei 4 esiti terminali. Se una classe
resta sotto quota (tipicamente off-road/timeout con un buon agente), aumenta
`N_ROLLOUT_STEPS` e ricampiona.

In [ ]:
buckets = {sid: [] for sid in names}
for traj, sid in zip(trajectories, terminal):
    if sid in buckets and len(buckets[sid]) < N_PER_CLASS:
        buckets[sid].append(traj)

balanced = [traj for sid in names for traj in buckets[sid]]
for sid, name in names.items():
    print(f"  {name:8s}: {len(buckets[sid])}/{N_PER_CLASS}")
assert all(len(b) == N_PER_CLASS for b in buckets.values()), \
    "Classi incomplete: aumenta N_ROLLOUT_STEPS."
print(f"Episodi bilanciati totali: {len(balanced)}")

## 5. Salvataggio

Lista piatta di episodi (`list[Trajectory]`), lo stesso formato di
`debug_dataset_full_ep.pkl`. La versione a transizioni piatte
(`debug_dataset.pkl`) si ottiene concatenando gli episodi; la conversione
inversa (da transizioni a episodi) è in `extract_debug_episodes.ipynb`.

In [ ]:
with open(OUT_PATH, "wb") as f:
    pickle.dump(balanced, f)
print(f"Salvato: {OUT_PATH} ({OUT_PATH.stat().st_size} bytes)")
env.close()